In [2]:
#!/usr/bin/env python
"""
════════════════════════════════════════════════════════════════════════
 NAFNET-BEAST — 128ch/8blk ON TPU v5e-8
════════════════════════════════════════════════════════════════════════
 The biggest NAFNet we've ever trained.
 50M params, batch 16, Charbonnier loss, TPU-optimized.
════════════════════════════════════════════════════════════════════════
"""

"\n════════════════════════════════════════════════════════════════════════\n NAFNET-BEAST — 128ch/8blk ON TPU v5e-8\n════════════════════════════════════════════════════════════════════════\n The biggest NAFNet we've ever trained.\n 50M params, batch 16, Charbonnier loss, TPU-optimized.\n════════════════════════════════════════════════════════════════════════\n"

In [3]:

COMP_DATA    = "/kaggle/input/competitions/ExeBit_kla_ai_hack/KLA AI - HACK"
TRAIN_GT     = COMP_DATA + "/train/GT"
TRAIN_LR     = COMP_DATA + "/train/NoisyLR"
WORK_DIR     = "/kaggle/working"

class CFG:
    # Architecture — THE BEAST
    ch            = 128
    n_blocks      = 12
    
    # Training Phase 1
    epochs_p1     = 80
    lr_p1         = 2e-4
    batch_p1      = 16
    
    # Training Phase 2 (fine-tune)
    epochs_p2     = 30
    lr_p2         = 5e-5
    batch_p2      = 8
    
    # General
    weight_decay  = 1e-4
    grad_clip     = 1.0
    val_frac      = 0.05
    
    # Save
    save_name     = "beast_deep_128ch_12blk.pth"

In [4]:

import os, glob, math, time, random, warnings, gc
import numpy as np

# TPU-specific imports — must come before torch in some setups
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

# PyTorch XLA for TPU
import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.distributed.parallel_loader as pl

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)

# TPU device
DEVICE = xm.xla_device()
print(f"Device: {DEVICE}")
print(f"TPU device ready: {DEVICE}")

os.makedirs(WORK_DIR, exist_ok=True)
print(f"Train GT:  {len(os.listdir(TRAIN_GT))} files")
print(f"Train LR:  {len(os.listdir(TRAIN_LR))} files")

/usr/local/lib/python3.12/site-packages/torch_xla/__init__.py:258: UserWarning: `tensorflow` can conflict with `torch-xla`. Prefer `tensorflow-cpu` when using PyTorch/XLA. To silence this warning, `pip uninstall -y tensorflow && pip install tensorflow-cpu`. If you are in a notebook environment such as Colab or Kaggle, restart your notebook runtime afterwards.
  warnings.warn(
E0000 00:00:1773687597.328439      12 common_lib.cc:648] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: === 
learning/45eac/tfrc/runtime/common_lib.cc:238


Device: xla:0
TPU device ready: xla:0
Train GT:  2400 files
Train LR:  2400 files


In [5]:
# NAFNet with Linear-based SCA (no Conv2d alignment issues)

class SimpleGate(nn.Module):
    def forward(self, x):
        x1, x2 = x.chunk(2, dim=1)
        return x1 * x2

class SimplifiedChannelAttention(nn.Module):
    """Linear-based SCA — works on any channel count, any device."""
    def __init__(self, ch):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Linear(ch, ch)
    def forward(self, x):
        b, c, _, _ = x.shape
        w = self.fc(self.pool(x).view(b, c)).view(b, c, 1, 1)
        return x * w

class NAFBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.norm = nn.GroupNorm(1, ch)
        self.body = nn.Sequential(
            nn.Conv2d(ch, ch * 2, 3, padding=1),
            SimpleGate(),
            nn.Conv2d(ch, ch, 3, padding=1),
        )
        self.sca   = SimplifiedChannelAttention(ch)
        self.scale = nn.Parameter(torch.zeros(1))
    def forward(self, x):
        res = self.body(self.norm(x))
        res = self.sca(res)
        return x + res * self.scale

class NAFNetBeast(nn.Module):
    """NAFNet U-Net with PixelShuffle ×2. Identical architecture, just bigger."""
    def __init__(self, ch=128, n_blocks=8):
        super().__init__()
        # Encoder
        self.head = nn.Conv2d(1, ch, 3, padding=1)
        self.enc1 = nn.Sequential(*[NAFBlock(ch) for _ in range(n_blocks)])
        self.down1 = nn.Conv2d(ch, ch*2, 2, stride=2)
        self.enc2 = nn.Sequential(*[NAFBlock(ch*2) for _ in range(n_blocks)])
        self.down2 = nn.Conv2d(ch*2, ch*4, 2, stride=2)
        # Bottleneck
        self.mid = nn.Sequential(*[NAFBlock(ch*4) for _ in range(n_blocks)])
        # Decoder
        self.up2 = nn.Sequential(nn.Conv2d(ch*4, ch*2 * 4, 1), nn.PixelShuffle(2))
        self.fuse2 = nn.Conv2d(ch*4, ch*2, 1)
        self.dec2 = nn.Sequential(*[NAFBlock(ch*2) for _ in range(n_blocks)])
        self.up1 = nn.Sequential(nn.Conv2d(ch*2, ch * 4, 1), nn.PixelShuffle(2))
        self.fuse1 = nn.Conv2d(ch*2, ch, 1)
        self.dec1 = nn.Sequential(*[NAFBlock(ch) for _ in range(n_blocks)])
        # 2× SR upsample
        self.tail = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1),
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(ch, 4, 3, padding=1),
            nn.PixelShuffle(2),
        )

    def forward(self, x):
        x_bic = F.interpolate(x, scale_factor=2, mode='bicubic', align_corners=False)
        e1 = self.enc1(self.head(x))
        e2 = self.enc2(self.down1(e1))
        m  = self.mid(self.down2(e2))
        d2 = self.fuse2(torch.cat([self.up2(m), e2], 1))
        d2 = self.dec2(d2)
        d1 = self.fuse1(torch.cat([self.up1(d2), e1], 1))
        d1 = self.dec1(d1)
        return self.tail(d1) + x_bic

In [6]:

class SRDataset(Dataset):
    def __init__(self, lr_dir, gt_dir, augment=True):
        self.lr_paths = sorted(glob.glob(os.path.join(lr_dir, "*.npy")))
        self.gt_paths = sorted(glob.glob(os.path.join(gt_dir, "*.npy")))
        assert len(self.lr_paths) == len(self.gt_paths), \
            f"Mismatch: {len(self.lr_paths)} LR vs {len(self.gt_paths)} GT"
        self.augment = augment

    def __len__(self):
        return len(self.lr_paths)

    def __getitem__(self, idx):
        lr = np.load(self.lr_paths[idx]).astype(np.float32)
        gt = np.load(self.gt_paths[idx]).astype(np.float32)

        if self.augment:
            if random.random() > 0.5:
                lr = np.fliplr(lr).copy(); gt = np.fliplr(gt).copy()
            if random.random() > 0.5:
                lr = np.flipud(lr).copy(); gt = np.flipud(gt).copy()
            k = random.randint(0, 3)
            lr = np.rot90(lr, k).copy(); gt = np.rot90(gt, k).copy()

        return torch.from_numpy(lr).unsqueeze(0), torch.from_numpy(gt).unsqueeze(0)

In [7]:

class CharbonnierLoss(nn.Module):
    def __init__(self, eps=1e-3):
        super().__init__()
        self.eps_sq = eps ** 2
    def forward(self, pred, target):
        return torch.mean(torch.sqrt((pred - target) ** 2 + self.eps_sq))

In [8]:

def compute_psnr_np(pred, gt):
    mse = np.mean((pred - gt) ** 2)
    if mse < 1e-12: return 60.0
    dr = gt.max() - gt.min()
    return 10 * math.log10(dr ** 2 / mse) if dr > 1e-8 else 0.0


def validate_tpu(model, val_pairs):
    """Validate on CPU to avoid TPU graph recompilation."""
    model_cpu = NAFNetBeast(ch=CFG.ch, n_blocks=CFG.n_blocks)
    # Copy weights to CPU
    state = {}
    for k, v in model.state_dict().items():
        state[k] = v.cpu()
    model_cpu.load_state_dict(state)
    model_cpu.eval()

    psnr_sum, cnt = 0.0, 0
    with torch.no_grad():
        for lr_np, gt_np in val_pairs:
            lr_t = torch.from_numpy(lr_np).float().unsqueeze(0).unsqueeze(0)
            pred = model_cpu(lr_t).squeeze().numpy()
            pred = np.clip(pred, 0, 1)
            psnr_sum += compute_psnr_np(pred, gt_np)
            cnt += 1

    del model_cpu
    return psnr_sum / max(cnt, 1)


def train_phase_tpu(model, dataloader, optimizer, scheduler, criterion,
                    val_pairs, epochs, phase_name, best_psnr=0, best_state=None):
    """TPU-optimized training loop."""
    save_path = os.path.join(WORK_DIR, CFG.save_name)
    t0 = time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        run_loss = 0.0
        n_batches = 0

        # TPU ParallelLoader wraps the dataloader
        para_loader = pl.ParallelLoader(dataloader, [DEVICE])
        loader = para_loader.per_device_loader(DEVICE)

        for lr_img, gt_img in loader:
            pred = model(lr_img)
            loss = criterion(pred, gt_img)

            optimizer.zero_grad()
            loss.backward()

            # TPU gradient clipping
            xm.reduce_gradients(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)

            # TPU-specific optimizer step
            xm.optimizer_step(optimizer)

            run_loss += loss.item()
            n_batches += 1

        scheduler.step()
        avg_loss = run_loss / max(n_batches, 1)

        # Validate every 2 epochs
        marker = ""
        if epoch % 2 == 0 or epoch <= 3 or epoch == epochs:
            vpsnr = validate_tpu(model, val_pairs)

            if vpsnr > best_psnr:
                best_psnr = vpsnr
                # Save to CPU state dict
                best_state = {}
                for k, v in model.state_dict().items():
                    best_state[k] = v.cpu().clone()
                # Save to disk immediately
                torch.save(best_state, save_path)
                marker = f" ★ saved"
        else:
            vpsnr = best_psnr  # placeholder for logging

        # Periodic backup
        if epoch % 10 == 0:
            backup_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            torch.save(backup_state, os.path.join(WORK_DIR, f"beast_ep{epoch}.pth"))

        elapsed = time.time() - t0
        if epoch <= 3 or epoch % 2 == 0 or marker:
            print(f"  {phase_name} Ep {epoch:3d}/{epochs}  loss={avg_loss:.5f}  "
                  f"psnr={vpsnr:.2f}  lr={scheduler.get_last_lr()[0]:.1e}  "
                  f"[{elapsed/60:.1f}m]{marker}")

    elapsed = time.time() - t0
    print(f"\n  {phase_name} done: {elapsed/60:.1f} min, best PSNR: {best_psnr:.3f} dB")
    return best_psnr, best_state

In [9]:

def main():
    print("\n" + "█" * 60)
    print("  NAFNET-BEAST — 128ch/8blk ON TPU")
    print("█" * 60)

    # ── Model ──
    model = NAFNetBeast(ch=CFG.ch, n_blocks=CFG.n_blocks).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Parameters: {n_params:,}")
    print(f"  Architecture: ch={CFG.ch}, blocks={CFG.n_blocks}")
    print(f"  Bottleneck: {CFG.ch * 4} channels")

    # ── Dataset ──
    full_ds = SRDataset(TRAIN_LR, TRAIN_GT, augment=True)
    n_val = max(4, int(CFG.val_frac * len(full_ds)))
    n_train = len(full_ds) - n_val
    train_ds, val_ds = random_split(
        full_ds, [n_train, n_val],
        generator=torch.Generator().manual_seed(SEED))

    # Preload validation pairs (CPU numpy)
    val_ds_noaug = SRDataset(TRAIN_LR, TRAIN_GT, augment=False)
    val_pairs = []
    for i in val_ds.indices:
        lr_t, gt_t = val_ds_noaug[i]
        val_pairs.append((lr_t.squeeze().numpy(), gt_t.squeeze().numpy()))
    print(f"  Train: {n_train}  |  Val: {n_val}")

    criterion = CharbonnierLoss().to(DEVICE)
    best_psnr = 0.0
    best_state = None

    # ════════════════════════════════════════════
    # PHASE 1
    # ════════════════════════════════════════════
    print(f"\n  Phase 1: {CFG.epochs_p1} epochs, lr={CFG.lr_p1}, batch={CFG.batch_p1}")

    train_dl = DataLoader(train_ds, batch_size=CFG.batch_p1, shuffle=True,
                          num_workers=4, drop_last=True)

    optimizer = optim.AdamW(model.parameters(), lr=CFG.lr_p1,
                            weight_decay=CFG.weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=CFG.epochs_p1, eta_min=1e-6)

    best_psnr, best_state = train_phase_tpu(
        model, train_dl, optimizer, scheduler, criterion,
        val_pairs, CFG.epochs_p1, "P1", best_psnr, best_state)

    # ════════════════════════════════════════════
    # PHASE 2
    # ════════════════════════════════════════════
    if CFG.epochs_p2 > 0 and best_state is not None:
        print(f"\n  Phase 2: {CFG.epochs_p2} epochs, lr={CFG.lr_p2}, batch={CFG.batch_p2}")

        # Load best weights from Phase 1
        model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

        ft_dl = DataLoader(train_ds, batch_size=CFG.batch_p2, shuffle=True,
                           num_workers=4, drop_last=True)

        optimizer = optim.AdamW(model.parameters(), lr=CFG.lr_p2,
                                weight_decay=CFG.weight_decay)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=CFG.epochs_p2, eta_min=1e-7)

        best_psnr, best_state = train_phase_tpu(
            model, ft_dl, optimizer, scheduler, criterion,
            val_pairs, CFG.epochs_p2, "P2", best_psnr, best_state)

    # ════════════════════════════════════════════
    # DONE
    # ════════════════════════════════════════════
    save_path = os.path.join(WORK_DIR, CFG.save_name)
    if best_state:
        torch.save(best_state, save_path)

    print(f"\n{'█' * 60}")
    print(f"  BEAST TRAINING COMPLETE")
    print(f"  Best PSNR: {best_psnr:.3f} dB")
    print(f"  Saved: {save_path}")
    print(f"  Params: {n_params:,}")

    # List all saved files
    print(f"\n  Files in {WORK_DIR}:")
    for f in sorted(os.listdir(WORK_DIR)):
        if f.endswith('.pth'):
            size = os.path.getsize(os.path.join(WORK_DIR, f)) / 1e6
            print(f"    {f} ({size:.1f} MB)")

    print(f"{'█' * 60}")

In [ ]:

if __name__ == "__main__" or True:
    main()


████████████████████████████████████████████████████████████
  NAFNET-BEAST — 128ch/8blk ON TPU
████████████████████████████████████████████████████████████
  Parameters: 144,853,568
  Architecture: ch=128, blocks=12
  Bottleneck: 512 channels
  Train: 2280  |  Val: 120

  Phase 1: 80 epochs, lr=0.0002, batch=16
  P1 Ep   1/80  loss=0.04232  psnr=27.05  lr=2.0e-04  [3.9m] ★ saved
  P1 Ep   2/80  loss=0.03507  psnr=27.57  lr=2.0e-04  [6.5m] ★ saved
  P1 Ep   3/80  loss=0.04246  psnr=27.32  lr=2.0e-04  [9.0m]
  P1 Ep   4/80  loss=0.03353  psnr=27.84  lr=2.0e-04  [11.5m] ★ saved
  P1 Ep   6/80  loss=0.03256  psnr=27.82  lr=2.0e-04  [15.7m]
  P1 Ep   8/80  loss=0.03186  psnr=28.40  lr=2.0e-04  [19.8m] ★ saved
  P1 Ep  10/80  loss=0.03086  psnr=28.54  lr=1.9e-04  [23.9m] ★ saved
  P1 Ep  12/80  loss=0.03062  psnr=28.65  lr=1.9e-04  [27.9m] ★ saved
  P1 Ep  14/80  loss=0.03053  psnr=28.66  lr=1.9e-04  [32.0m] ★ saved
  P1 Ep  16/80  loss=0.03033  psnr=28.73  lr=1.8e-04  [36.1m] ★ saved
  P1

In [ ]:
print("Hello world")